# OPRB primary benchmark: granite
Shared tested pipeline. Colab: one GPU; Kaggle: set GPUS = "0,1" for T4 x2.
Use a fresh session per model. LIMIT=5 is a smoke test; None selects all test pages.
Download results before ending the session. Reproducibility is documented, not guaranteed bitwise.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
FILES = {'pyproject.toml': '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "oprb-lab"\nversion = "0.1.0"\ndescription = "Pretrained OCR and VLM evaluation on occluded OPRB pages"\nrequires-python = ">=3.10"\ndependencies = ["Pillow>=10", "huggingface-hub>=0.30", "rapidfuzz>=3", "pandas>=2", "matplotlib>=3.8", "tqdm>=4.66"]\n\n[project.optional-dependencies]\nvlm = ["transformers>=4.57.1,<5", "accelerate>=1", "docling-core>=2.48", "einops", "timm>=1", "sentencepiece"]\npaddle = ["paddleocr>=3.2,<4"]\ntest = ["pytest>=8"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.pytest.ini_options]\npythonpath = ["src"]\ntestpaths = ["tests"]\n', 'src/oprb_lab/__init__.py': '"""OPRB pretrained-model benchmark."""\n', 'src/oprb_lab/__main__.py': 'import argparse\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom importlib.metadata import version, PackageNotFoundError\nfrom pathlib import Path\n\nfrom .data import collect, download\nfrom .models import MODELS, PROMPT\nfrom .report import summarize\nfrom .storage import fingerprint\n\n\ndef launch(args):\n    from huggingface_hub import model_info\n    samples = collect(args.data, args.limit, args.seed)\n    gpus = args.gpus.split(",")\n    if any(not g.isdigit() for g in gpus) or len(set(gpus)) != len(gpus):\n        raise ValueError("gpus must be unique numeric IDs, e.g. 0,1")\n    revision = model_info(MODELS[args.model], revision=args.revision).sha if args.model != "paddle" else "paddleocr-package-assets"\n    packages = {}\n    for name in ("torch", "torchvision", "transformers", "paddleocr", "paddlepaddle-gpu", "docling-core", "Pillow", "rapidfuzz"):\n        try:\n            packages[name] = version(name)\n        except PackageNotFoundError:\n            pass\n    config = dict(schema=2, model=args.model, model_id=MODELS[args.model], revision=revision, packages=packages,\n                  precision="float16" if args.model != "paddle" else "PaddleOCR default float32",\n                  max_tokens=args.max_tokens, max_pixels=args.max_pixels, max_tiles=args.max_tiles,\n                  prompt="Convert this page to docling." if args.model == "granite" else "GOT processor unformatted OCR" if args.model == "got" else PROMPT,\n                  limit=args.limit, seed=args.seed, world_size=len(gpus),\n                  manifest_sha=fingerprint([{k: v for k, v in s.items() if k not in ("image_path", "gt_path")} for s in samples]),\n                  normalization="NFKC whitespace casefold", gt_order="annotation order",\n                  code_sha=fingerprint({p.name: p.read_text(encoding="utf-8") for p in Path(__file__).parent.glob("*.py")}))\n    root = Path(args.output) / f"{args.model}-{fingerprint(config)}"\n    root.mkdir(parents=True, exist_ok=True)\n    (root / "config.json").write_text(json.dumps(config, indent=2))\n    (root / "manifest.json").write_text(json.dumps(samples, indent=2))\n    workers, logs = [], []\n    start = time.perf_counter()\n    try:\n        for rank, gpu in enumerate(gpus):\n            env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="2",\n                       PYTHONHASHSEED=str(args.seed), CUBLAS_WORKSPACE_CONFIG=":4096:8")\n            env.pop("LD_LIBRARY_PATH", None)\n            log = (root / f"rank{rank}.log").open("a", encoding="utf-8")\n            logs.append(log)\n            workers.append(subprocess.Popen([sys.executable, "-u", "-m", "oprb_lab.worker", str(root), str(rank), str(len(gpus))],\n                                             env=env, stdout=log, stderr=subprocess.STDOUT))\n        print(f"Running {len(workers)} workers. Logs/results: {root}", flush=True)\n        while any(w.poll() is None for w in workers):\n            counts = {f"rank{rank}": sum(1 for _ in p.open(encoding="utf-8")) if p.exists() else 0\n                      for rank in range(len(workers)) for p in [root / f"rank{rank}.jsonl"]}\n            print(f"Saved rows: {counts}", flush=True)\n            time.sleep(10)\n    finally:\n        for worker in workers:\n            if worker.poll() is None:\n                worker.terminate()\n        for worker in workers:\n            worker.wait()\n        for log in logs:\n            log.close()\n    (root / "session.json").write_text(json.dumps(dict(wall_sec=time.perf_counter()-start,\n                                                        exit_codes=[w.returncode for w in workers]), indent=2))\n    summary = summarize(root)\n    if any(w.returncode for w in workers) or not summary["complete"]:\n        raise SystemExit("Incomplete evaluation: inspect logs; rerun the same command to retry errors/missing pages.")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="OPRB pretrained OCR benchmark")\n    sub = parser.add_subparsers(dest="command", required=True)\n    fetch = sub.add_parser("download")\n    fetch.add_argument("--data", default="data/oprb")\n    fetch.add_argument("--revision", default="main", help="Dataset revision; resolved to a commit before downloading")\n    report = sub.add_parser("summarize")\n    report.add_argument("run_dir")\n    run = sub.add_parser("run")\n    run.add_argument("--model", choices=MODELS, required=True)\n    run.add_argument("--data", default="data/oprb")\n    run.add_argument("--output", default="results")\n    run.add_argument("--gpus", default="0,1")\n    run.add_argument("--limit", type=int, default=None, help="Pages per category; omit for full test")\n    run.add_argument("--seed", type=int, default=42)\n    run.add_argument("--revision", default="main")\n    run.add_argument("--max-tokens", type=int, default=4096)\n    run.add_argument("--max-pixels", type=int, default=1280*28*28)\n    run.add_argument("--max-tiles", type=int, default=12)\n    args = parser.parse_args()\n    if args.command == "download":\n        download(args.data, args.revision)\n    elif args.command == "summarize":\n        summarize(args.run_dir)\n    else:\n        if min(args.max_tokens, args.max_pixels, args.max_tiles) < 1:\n            parser.error("Token, pixel, and tile limits must be positive")\n        launch(args)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/oprb_lab/comparison.py': '"""Compare the five registered models on matched category-qualified pages."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom .models import MODELS\nfrom .report import METRICS\nfrom .storage import read_rows\n\n\ndef compare(run_dirs, output):\n    import pandas as pd\n    runs, manifests, configs, frames = {}, [], [], []\n    for directory in map(Path, run_dirs):\n        config = json.loads((directory / "config.json").read_text())\n        key = config["model"]\n        if key not in MODELS or key in runs:\n            raise ValueError("Supply exactly one run per registered model")\n        runs[key] = directory\n        configs.append(config)\n        manifest = json.loads((directory / "manifest.json").read_text())\n        expected = {s["uid"] for s in manifest}\n        if len(expected) != len(manifest):\n            raise ValueError("Duplicate manifest UIDs")\n        manifests.append(expected)\n        rows = {}\n        for path in sorted(directory.glob("rank[0-9]*.jsonl")):\n            for row in read_rows(path):\n                if row["uid"] in expected:\n                    rows[row["uid"]] = dict(row, model=key)\n        frames.append(pd.DataFrame(rows.values(), columns=None if rows else ["uid", "status", "model"]))\n    if set(runs) != set(MODELS):\n        raise ValueError(f"All five models are required: {list(MODELS)}")\n    if any(ids != manifests[0] for ids in manifests):\n        raise ValueError("Runs use different sample manifests")\n    for field in ("manifest_sha", "normalization", "gt_order", "seed", "code_sha"):\n        if any(c.get(field) != configs[0].get(field) for c in configs):\n            raise ValueError(f"Incompatible evaluation protocol: {field}")\n    data = pd.concat(frames, ignore_index=True)\n    coverage = []\n    common = set(manifests[0])\n    for key in MODELS:\n        subset = data[data.model == key]\n        successful = set(subset.loc[subset.status == "ok", "uid"])\n        common &= successful\n        coverage.append(dict(model=key, expected=len(manifests[0]), success=len(successful),\n                             errors=int((subset.status == "error").sum()),\n                             missing=len(manifests[0] - set(subset.uid))))\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(coverage).to_csv(output / "coverage.csv", index=False)\n    if not common:\n        raise ValueError("No common successful pages; inspect coverage.csv")\n    paired = data[(data.status == "ok") & data.uid.isin(common)].copy()\n    # Comparing GT also catches inconsistent annotations despite equal sample names.\n    if paired.groupby("uid").ground_truth.nunique(dropna=False).gt(1).any():\n        raise ValueError("Ground truth differs across runs")\n    table = paired.groupby("model")[METRICS].mean()\n    table["n_common"] = len(common)\n    table["pareto"] = [not any((other.latency_sec <= row.latency_sec and other.wer <= row.wer)\n                       and (other.latency_sec < row.latency_sec or other.wer < row.wer)\n                       for _, other in table.iterrows()) for _, row in table.iterrows()]\n    table.to_csv(output / "comparison.csv")\n    (output / "comparison.tex").write_text(table.to_latex(float_format="%.4f"), encoding="utf-8")\n    paired.to_csv(output / "paired_pages.csv", index=False)\n    for group in ("category", "occlusion_type", "density"):\n        paired.groupby(["model", group])[METRICS].mean().to_csv(output / f"by_{group}.csv")\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots()\n    for key, row in table.iterrows():\n        ax.scatter(row.latency_sec, row.wer)\n        ax.annotate(key, (row.latency_sec, row.wer))\n    ax.set(xlabel="Mean end-to-end latency (s/page)", ylabel="Macro WER", title="Matched-page comparison")\n    fig.tight_layout()\n    fig.savefig(output / "latency_wer.png", dpi=180)\n    plt.close(fig)\n    (output / "protocol.json").write_text(json.dumps(dict(\n        runs={k: str(v) for k, v in runs.items()}, configs=configs,\n        common_uids=sorted(common), note="Report coverage alongside paired scores. Compare latency only on equivalent hardware and GPU concurrency."), indent=2))\n    return table\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("runs", nargs="+", help="Exactly five fingerprinted run directories")\n    parser.add_argument("--output", default="comparison")\n    args = parser.parse_args()\n    print(compare(args.runs, args.output))\n', 'src/oprb_lab/data.py': '"""Download only occluded test images and scoring annotations."""\nimport random\nimport hashlib\nimport shutil\nimport tarfile\nfrom pathlib import Path\n\nEXPECTED = {"text": 6000, "nontext": 3000}\n\n\ndef extract_flat(archive, destination, suffix):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    seen = set()\n    with tarfile.open(archive, "r:gz") as tar:\n        for item in tar:\n            if not item.isfile() or not item.name.lower().endswith(suffix):\n                continue\n            name = item.name.replace("\\\\", "/").split("/")[-1]\n            if name in seen:\n                raise ValueError(f"Duplicate archive basename: {name}")\n            seen.add(name)\n            with tar.extractfile(item) as source, (destination / name).open("wb") as target:\n                shutil.copyfileobj(source, target)\n\n\ndef download(root, revision="main"):\n    from huggingface_hub import hf_hub_download, dataset_info\n    revision = dataset_info("kpurkayastha/OPRB", revision=revision).sha\n    for category in EXPECTED:\n        for kind, suffix in (("img", ".jpg"), ("gt", ".txt")):\n            destination = Path(root) / category / kind\n            marker = destination / ".complete"\n            if marker.exists() and marker.read_text() == revision and len(list(destination.glob(f"*{suffix}"))) == EXPECTED[category]:\n                continue\n            archive = hf_hub_download("kpurkayastha/OPRB", f"test/{category}/{kind}.tar.gz", repo_type="dataset", revision=revision)\n            extract_flat(archive, destination, suffix)\n            count = len(list(destination.glob(f"*{suffix}")))\n            if count != EXPECTED[category]:\n                raise ValueError(f"{category}/{kind}: expected {EXPECTED[category]}, got {count}")\n            marker.write_text(revision)\n\n\ndef file_sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef collect(root, limit=None, seed=42):\n    if limit is not None and limit < 1:\n        raise ValueError("limit must be positive or None")\n    samples = []\n    for category in EXPECTED:\n        images = {p.stem: p for p in (Path(root) / category / "img").glob("*.jpg")}\n        labels = {p.stem: p for p in (Path(root) / category / "gt").glob("*.txt")}\n        if not images or images.keys() != labels.keys():\n            raise ValueError(f"Missing or unmatched images/GT in {category}")\n        if limit is None and len(images) != EXPECTED[category]:\n            raise ValueError(f"Full test requires {EXPECTED[category]} pairs in {category}; got {len(images)}")\n        keys = sorted(images)\n        if limit is not None:\n            keys = sorted(random.Random(seed).sample(keys, min(limit, len(keys))))\n        for key in keys:\n            parts = key.split("__", 2)\n            samples.append(dict(uid=f"{category}/{key}", sample_id=key, category=category,\n                                image_path=str(images[key].resolve()), gt_path=str(labels[key].resolve()),\n                                image_sha=file_sha(images[key]), gt_sha=file_sha(labels[key]),\n                                occlusion_type=parts[0], density=parts[1] if len(parts) > 1 else "unknown"))\n    return samples\n\n\ndef read_gt(path):\n    # Keep annotation order: this policy must be disclosed for multi-column pages.\n    tokens = []\n    for line in Path(path).read_text(encoding="utf-8-sig").splitlines():\n        fields = line.split("\\t")\n        if len(fields) >= 11 and fields[0].strip():\n            tokens.append(fields[0].strip())\n    if not tokens:\n        raise ValueError(f"No GT tokens: {path}")\n    return " ".join(tokens)\n\n\ndef shard(samples, rank, world_size):\n    if world_size < 1 or not 0 <= rank < world_size:\n        raise ValueError("Invalid rank/world_size")\n    return samples[rank::world_size]\n', 'src/oprb_lab/metrics.py': 'import re\nimport unicodedata\nfrom collections import Counter\n\n\ndef normalize(text):\n    return re.sub(r"\\s+", " ", unicodedata.normalize("NFKC", text)).strip().casefold()\n\n\ndef score(reference, prediction):\n    from rapidfuzz.distance import Levenshtein\n    ref, hyp = normalize(reference), normalize(prediction)\n    if not ref:\n        raise ValueError("Empty reference")\n    a, b = Counter(ref.split()), Counter(hyp.split())\n    matched = sum((a & b).values())\n    precision = matched / sum(b.values()) if b else 0.0\n    recall = matched / sum(a.values())\n    return dict(cer=Levenshtein.distance(ref, hyp) / len(ref),\n                wer=Levenshtein.distance(ref.split(), hyp.split()) / len(ref.split()),\n                token_precision=precision, token_recall=recall,\n                token_f1=2 * precision * recall / (precision + recall) if precision + recall else 0.0)\n', 'src/oprb_lab/models.py': '"""Image-only adapters. Ground truth is never provided to a model."""\nfrom PIL import Image\n\nMODELS = {\n    "paddle": "PP-OCRv5_mobile_det+PP-OCRv5_mobile_rec",\n    "granite": "ibm-granite/granite-docling-258M",\n    "got": "stepfun-ai/GOT-OCR-2.0-hf",\n    "internvl": "OpenGVLab/InternVL2_5-1B",\n    "qwen": "Qwen/Qwen3-VL-2B-Instruct",\n}\nPROMPT = ("Transcribe all text visible on this document page in natural reading order. "\n          "Return only the transcription. Do not explain, describe, or invent unreadable text.")\n\n\ndef intern_tiles(image, max_tiles=12, size=448):\n    """Dynamic grid + thumbnail, adapted from the official InternVL model card."""\n    w, h = image.size\n    ratios = sorted({(i, j) for i in range(1, max_tiles + 1)\n                     for j in range(1, max_tiles + 1) if i * j <= max_tiles},\n                    key=lambda p: (p[0] * p[1], p[0], p[1]))\n    best, distance = (1, 1), float("inf")\n    for ratio in ratios:\n        diff = abs(w / h - ratio[0] / ratio[1])\n        if diff < distance or (diff == distance and w * h > .5 * size * size * ratio[0] * ratio[1]):\n            best, distance = ratio, diff\n    cols, rows = best\n    resized = image.resize((cols * size, rows * size))\n    tiles = [resized.crop((x * size, y * size, (x + 1) * size, (y + 1) * size))\n             for y in range(rows) for x in range(cols)]\n    if len(tiles) > 1:\n        tiles.append(image.resize((size, size)))\n    return tiles\n\n\nclass Adapter:\n    def __init__(self, name, revision, max_tokens, max_pixels, max_tiles):\n        import torch\n        self.name, self.max_tokens, self.max_tiles = name, max_tokens, max_tiles\n        self.torch = torch\n        self.device = "cuda:0"  # Worker sees only its assigned physical GPU.\n        self.dtype = torch.float16\n        self.parameter_count = None\n        if name == "paddle":\n            from paddleocr import PaddleOCR\n            self.model = PaddleOCR(\n                text_detection_model_name="PP-OCRv5_mobile_det",\n                text_recognition_model_name="PP-OCRv5_mobile_rec",\n                use_doc_orientation_classify=False, use_doc_unwarping=False,\n                use_textline_orientation=False, device="gpu:0")\n            return\n        from transformers import AutoProcessor, AutoModelForImageTextToText\n        model_id = MODELS[name]\n        options = dict(revision=revision, torch_dtype=self.dtype, low_cpu_mem_usage=True)\n        if name == "internvl":\n            from transformers import AutoModel, AutoTokenizer\n            self.model = AutoModel.from_pretrained(model_id, trust_remote_code=True,\n                                                  use_flash_attn=False, **options).to(self.device).eval()\n            self.tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision,\n                                                           trust_remote_code=True, use_fast=False)\n        else:\n            processor_options = {"max_pixels": max_pixels} if name == "qwen" else {}\n            self.processor = AutoProcessor.from_pretrained(model_id, revision=revision, **processor_options)\n            self.model = AutoModelForImageTextToText.from_pretrained(\n                model_id, attn_implementation="sdpa", **options).to(self.device).eval()\n        self.parameter_count = sum(p.numel() for p in self.model.parameters())\n\n    def predict(self, path):\n        if self.name == "paddle":\n            results = list(self.model.predict(str(path)))\n            texts = []\n            for result in results:\n                # PaddleOCR Results expose the raw prediction as a dictionary.\n                texts.extend(result["rec_texts"])\n            return dict(prediction="\\n".join(texts), raw_prediction="\\n".join(texts),\n                        generated_tokens=None, possibly_truncated=False)\n        with Image.open(path) as source:\n            image = source.convert("RGB")\n        with self.torch.inference_mode():\n            if self.name == "internvl":\n                from torchvision.transforms import Compose, ToTensor, Normalize\n                transform = Compose([ToTensor(), Normalize((.485, .456, .406), (.229, .224, .225))])\n                pixels = self.torch.stack([transform(tile) for tile in intern_tiles(image, self.max_tiles)])\n                pixels = pixels.to(device=self.device, dtype=self.dtype)\n                raw = self.model.chat(self.tokenizer, pixels, "<image>\\n" + PROMPT,\n                                      dict(max_new_tokens=self.max_tokens, do_sample=False), history=None)\n                # chat returns decoded text, not raw generated token IDs.\n                return dict(prediction=raw, raw_prediction=raw, generated_tokens=None,\n                            possibly_truncated=None)\n            if self.name == "got":\n                inputs = self.processor(image, return_tensors="pt", format=False)\n            else:\n                text = "Convert this page to docling." if self.name == "granite" else PROMPT\n                messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]\n                prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n                inputs = self.processor(text=[prompt], images=[image], return_tensors="pt")\n            inputs = inputs.to(self.device)\n            for key, tensor in inputs.items():\n                if tensor.is_floating_point():\n                    inputs[key] = tensor.to(self.dtype)\n            output = self.model.generate(**inputs, max_new_tokens=self.max_tokens, do_sample=False)\n            tokens = output[0, inputs["input_ids"].shape[-1]:]\n            raw = self.processor.decode(tokens, skip_special_tokens=self.name != "granite",\n                                        clean_up_tokenization_spaces=False)\n            if self.name == "granite":\n                from docling_core.types.doc import DoclingDocument\n                from docling_core.types.doc.document import DocTagsDocument\n                doc = DoclingDocument.load_from_doctags(\n                    DocTagsDocument.from_doctags_and_image_pairs([raw], [image]), document_name="page")\n                prediction = doc.export_to_text()\n            else:\n                prediction = raw.strip()\n            return dict(prediction=prediction, raw_prediction=raw, generated_tokens=len(tokens),\n                        possibly_truncated=len(tokens) >= self.max_tokens)\n', 'src/oprb_lab/report.py': 'import json\nfrom pathlib import Path\n\nfrom .storage import read_rows\n\nMETRICS = ["cer", "wer", "token_precision", "token_recall", "token_f1", "latency_sec"]\n\n\ndef summarize(run_dir):\n    import pandas as pd\n    root = Path(run_dir)\n    manifest = json.loads((root / "manifest.json").read_text())\n    expected = {s["uid"] for s in manifest}\n    rows = {}\n    for path in sorted(root.glob("rank[0-9]*.jsonl")):\n        for row in read_rows(path):\n            if row["uid"] in expected:\n                rows[row["uid"]] = row\n    df = pd.DataFrame(rows.values()) if rows else pd.DataFrame(columns=["uid", "status"])\n    df.to_csv(root / "per_image_results.csv", index=False)\n    ok = df.loc[df.status == "ok"]\n    summary = dict(expected=len(expected), observed=len(df), success=len(ok),\n                   errors=int((df.status == "error").sum()), missing=len(expected - rows.keys()),\n                   complete=len(ok) == len(expected), aggregation="macro mean over successful pages")\n    if len(ok):\n        summary.update({m: float(ok[m].mean()) for m in METRICS})\n        summary["possibly_truncated"] = int(ok.possibly_truncated.fillna(False).sum())\n        summary["truncation_unknown"] = int(ok.possibly_truncated.isna().sum())\n        for groups, name in [(["category"], "category"), (["category", "occlusion_type"], "type"),\n                             (["category", "density"], "density")]:\n            ok.groupby(groups, dropna=False).agg(n=("uid", "count"), **{m: (m, "mean") for m in METRICS}).reset_index().to_csv(\n                root / f"summary_by_{name}.csv", index=False)\n        import matplotlib\n        matplotlib.use("Agg")\n        import matplotlib.pyplot as plt\n        ax = ok.groupby("category")[["cer", "wer"]].mean().plot.bar(rot=0)\n        ax.set_ylabel("Macro error rate (lower is better)")\n        plt.tight_layout()\n        plt.savefig(root / "error_rates.png", dpi=160)\n        plt.close()\n    (root / "summary.json").write_text(json.dumps(summary, indent=2))\n    print(json.dumps(summary, indent=2))\n    return summary\n', 'src/oprb_lab/storage.py': 'import hashlib\nimport json\nfrom pathlib import Path\n\n\ndef fingerprint(config):\n    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16]\n\n\ndef read_rows(path):\n    path = Path(path)\n    if not path.exists():\n        return []\n    lines = path.read_text(encoding="utf-8").splitlines()\n    rows = []\n    for i, line in enumerate(lines):\n        try:\n            rows.append(json.loads(line))\n        except json.JSONDecodeError:\n            if i != len(lines) - 1:\n                raise\n            # Interrupted append: caller rewrites valid records before resuming.\n    return rows\n\n\ndef rewrite_rows(path, rows):\n    path = Path(path)\n    tmp = path.with_suffix(".tmp")\n    tmp.write_text("".join(json.dumps(r, ensure_ascii=False) + "\\n" for r in rows), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef resume_rows(path, selected):\n    """Keep only successful rows for this shard; errors remain eligible for retry."""\n    ids = {sample["uid"] for sample in selected}\n    completed = {row["uid"]: row for row in read_rows(path)\n                 if row.get("status") == "ok" and row["uid"] in ids}\n    rewrite_rows(path, list(completed.values()))\n    return [sample for sample in selected if sample["uid"] not in completed]\n', 'src/oprb_lab/worker.py': '"""Independent GPU worker; launch via the CLI, never fork a CUDA context."""\nimport argparse\nimport json\nimport os\nimport time\nimport random\nimport platform\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nfrom .data import read_gt, shard\nfrom .metrics import score\nfrom .models import Adapter\nfrom .storage import resume_rows\n\n\ndef run(run_dir, rank, world_size):\n    # CUDA_VISIBLE_DEVICES is set by the parent before importing any GPU framework.\n    import torch\n    root = Path(run_dir)\n    config = json.loads((root / "config.json").read_text())\n    random.seed(config["seed"])\n    import numpy as np\n    np.random.seed(config["seed"])\n    torch.manual_seed(config["seed"])\n    torch.cuda.manual_seed_all(config["seed"])\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True, warn_only=True)\n    if config["model"] == "paddle":\n        import paddle\n        paddle.seed(config["seed"])\n    samples = json.loads((root / "manifest.json").read_text())\n    selected = shard(samples, rank, world_size)\n    path = root / f"rank{rank}.jsonl"\n    todo = resume_rows(path, selected)\n    if not todo:\n        return\n    model = Adapter(config["model"], config["revision"], config["max_tokens"],\n                    config["max_pixels"], config["max_tiles"])\n    def synchronize():\n        if config["model"] == "paddle":\n            import paddle\n            paddle.device.cuda.synchronize()\n        else:\n            torch.cuda.synchronize()\n    # Warm-up is intentionally outside measured latency. Failure stops the worker.\n    model.predict(todo[0]["image_path"])\n    synchronize()\n    metadata = dict(rank=rank, physical_gpu=os.environ.get("CUDA_VISIBLE_DEVICES"),\n                    gpu=torch.cuda.get_device_name(0), parameter_count=model.parameter_count,\n                    torch=version("torch"), python=platform.python_version(), platform=platform.platform(),\n                    cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),\n                    reproducibility="Seeded; deterministic algorithms warn on unsupported operations; not a bitwise guarantee")\n    if config["model"] != "paddle":\n        metadata["transformers"] = version("transformers")\n    else:\n        metadata["paddleocr"] = version("paddleocr")\n    (root / f"rank{rank}_metadata.json").write_text(json.dumps(metadata, indent=2))\n    for i, sample in enumerate(todo):\n        row = dict(sample)\n        try:\n            synchronize()\n            start = time.perf_counter()\n            output = model.predict(sample["image_path"])\n            synchronize()\n            elapsed = time.perf_counter() - start\n            gt = read_gt(sample["gt_path"])\n            row.update(status="ok", latency_sec=elapsed, ground_truth=gt,\n                       **output, **score(gt, output["prediction"]))\n        except Exception as error:\n            row.update(status="error", error=repr(error))\n            print(f"rank={rank} ERROR {sample[\'uid\']}: {error}", flush=True)\n            # Do not change resolution or precision silently after OOM.\n            if isinstance(error, torch.cuda.OutOfMemoryError):\n                torch.cuda.empty_cache()\n        with path.open("a", encoding="utf-8") as stream:\n            stream.write(json.dumps(row, ensure_ascii=False) + "\\n")\n            stream.flush()\n            os.fsync(stream.fileno())\n        print(f"rank={rank} {i+1}/{len(todo)} {row[\'status\']}", flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("run_dir")\n    parser.add_argument("rank", type=int)\n    parser.add_argument("world_size", type=int)\n    args = parser.parse_args()\n    run(args.run_dir, args.rank, args.world_size)\n', 'tests/test_comparison.py': 'import json\n\nimport pytest\n\nfrom oprb_lab.comparison import compare\nfrom oprb_lab.metrics import score\nfrom oprb_lab.models import MODELS\nfrom oprb_lab.storage import rewrite_rows, resume_rows, read_rows\n\n\ndef make_runs(tmp_path):\n    runs = []\n    for key in MODELS:\n        path = tmp_path / key\n        path.mkdir()\n        (path / "config.json").write_text(json.dumps(dict(model=key, manifest_sha="same", normalization="nfkc", seed=42, code_sha="same", gt_order="annotation")))\n        (path / "manifest.json").write_text(json.dumps([dict(uid="text/a"), dict(uid="nontext/a")]))\n        rows = [dict(uid=f"{cat}/a", category=cat, status="ok", ground_truth="cat", prediction="cat",\n                     occlusion_type="Ink", density="2", latency_sec=1., **score("cat", "cat")) for cat in ("text", "nontext")]\n        rewrite_rows(path / "rank0.jsonl", rows)\n        runs.append(path)\n    return runs\n\n\ndef test_comparison_uses_common_success_and_reports_failure(tmp_path):\n    runs = make_runs(tmp_path)\n    rows = read_rows(runs[0] / "rank0.jsonl")\n    rows[1] = dict(uid="nontext/a", status="error", error="OOM")\n    rewrite_rows(runs[0] / "rank0.jsonl", rows)\n    table = compare(runs, tmp_path / "output")\n    assert set(table.index) == set(MODELS)\n    assert (table.n_common == 1).all()\n    import pandas as pd\n    coverage = pd.read_csv(tmp_path / "output/coverage.csv").set_index("model")\n    assert coverage.loc["paddle", "errors"] == 1\n    assert coverage.loc["paddle", "expected"] == 2\n\n\n@pytest.mark.parametrize("field", ["manifest_sha", "normalization", "code_sha", "seed"])\ndef test_comparison_rejects_changed_protocol(tmp_path, field):\n    runs = make_runs(tmp_path)\n    path = runs[0] / "config.json"\n    config = json.loads(path.read_text())\n    config[field] = "changed"\n    path.write_text(json.dumps(config))\n    with pytest.raises(ValueError, match="protocol"):\n        compare(runs, tmp_path / "output")\n\n\ndef test_comparison_requires_all_five(tmp_path):\n    with pytest.raises(ValueError, match="All five"):\n        compare(make_runs(tmp_path)[:4], tmp_path / "output")\n\n\ndef test_comparison_rejects_inconsistent_annotations(tmp_path):\n    runs = make_runs(tmp_path)\n    path = runs[0] / "rank0.jsonl"\n    rows = read_rows(path)\n    rows[0]["ground_truth"] = "different"\n    rewrite_rows(path, rows)\n    with pytest.raises(ValueError, match="Ground truth"):\n        compare(runs, tmp_path / "output")\n\n\ndef test_comparison_no_common_success_writes_coverage(tmp_path):\n    runs = make_runs(tmp_path)\n    rewrite_rows(runs[0] / "rank0.jsonl", [])\n    with pytest.raises(ValueError, match="No common"):\n        compare(runs, tmp_path / "output")\n    assert (tmp_path / "output/coverage.csv").exists()\n\n\ndef test_resume_retries_errors_and_qualifies_category(tmp_path):\n    path = tmp_path / "rank0.jsonl"\n    selected = [dict(uid="text/a"), dict(uid="nontext/a")]\n    rewrite_rows(path, [dict(uid="text/a", status="ok"), dict(uid="nontext/a", status="error"), dict(uid="other", status="ok")])\n    assert resume_rows(path, selected) == [dict(uid="nontext/a")]\n    assert read_rows(path) == [dict(uid="text/a", status="ok")]\n', 'tests/test_core.py': 'import io\nimport json\nimport tarfile\n\nimport pytest\nfrom PIL import Image\n\nfrom oprb_lab.data import collect, extract_flat, read_gt, shard\nfrom oprb_lab.metrics import normalize, score\nfrom oprb_lab.models import intern_tiles\nfrom oprb_lab.storage import fingerprint, read_rows, rewrite_rows\n\n\ndef test_metrics_duplicates_and_insertions():\n    result = score("cat cat dog", "cat dog dog dog")\n    assert result["token_precision"] == .5\n    assert result["token_recall"] == pytest.approx(2/3)\n    assert score("a", "a b c")["wer"] == 2\n    assert score("abc", "")["cer"] == 1\n    assert score("abc", "abc")["cer"] == 0\n    assert score("The cat sat", "The cat")["wer"] == pytest.approx(1/3)\n    with pytest.raises(ValueError):\n        score("", "abc")\n\n\ndef test_normalization():\n    assert normalize("  Ａ\\nB\\t c ") == "a b c"\n\n\ndef test_shards_are_disjoint_complete():\n    samples = list(range(13))\n    shards = [shard(samples, i, 2) for i in range(2)]\n    assert set(shards[0]).isdisjoint(shards[1])\n    assert sorted(shards[0] + shards[1]) == samples\n    with pytest.raises(ValueError):\n        shard(samples, 2, 2)\n\n\ndef test_dataset_pairs_gt_and_seed(tmp_path):\n    for category in ("text", "nontext"):\n        for kind in ("img", "gt"):\n            (tmp_path / category / kind).mkdir(parents=True)\n        for i in range(5):\n            stem = f"Black_Ink__2p0pct__{i}"\n            Image.new("RGB", (10, 10)).save(tmp_path / category / "img" / f"{stem}.jpg")\n            (tmp_path / category / "gt" / f"{stem}.txt").write_text("word\\t" + "\\t".join(["0"] * 10))\n    samples = collect(tmp_path, 2, 42)\n    assert len(samples) == 4\n    assert samples == collect(tmp_path, 2, 42)\n    assert len({s["uid"] for s in samples}) == 4\n    assert samples[0]["density"] == "2p0pct"\n    assert read_gt(samples[0]["gt_path"]) == "word"\n    original = samples[0]\n    from pathlib import Path\n    Path(original["gt_path"]).write_text("changed\\t" + "\\t".join(["0"] * 10))\n    updated = next(s for s in collect(tmp_path, 2, 42) if s["uid"] == original["uid"])\n    assert updated["gt_sha"] != original["gt_sha"]\n    assert updated["image_sha"] == original["image_sha"]\n    with pytest.raises(ValueError, match="Full test requires"):\n        collect(tmp_path)\n    (tmp_path / "text" / "gt" / "extra.txt").write_text("")\n    with pytest.raises(ValueError):\n        collect(tmp_path)\n\n\ndef test_archive_paths_flattened_and_filtered(tmp_path):\n    archive = tmp_path / "data.tar.gz"\n    with tarfile.open(archive, "w:gz") as tar:\n        for name in ("../../escape.txt", "folder/keep.txt", "ignore.jpg"):\n            info = tarfile.TarInfo(name)\n            info.size = 3\n            tar.addfile(info, io.BytesIO(b"abc"))\n    dest = tmp_path / "out"\n    extract_flat(archive, dest, ".txt")\n    assert sorted(p.name for p in dest.iterdir()) == ["escape.txt", "keep.txt"]\n    assert not (tmp_path / "escape.txt").exists()\n\n\ndef test_archive_duplicate_rejected(tmp_path):\n    archive = tmp_path / "data.tar.gz"\n    with tarfile.open(archive, "w:gz") as tar:\n        for name in ("a/x.txt", "b/x.txt"):\n            info = tarfile.TarInfo(name)\n            info.size = 1\n            tar.addfile(info, io.BytesIO(b"a"))\n    with pytest.raises(ValueError):\n        extract_flat(archive, tmp_path / "out", ".txt")\n\n\ndef test_resume_recovers_partial_last_line(tmp_path):\n    path = tmp_path / "rank0.jsonl"\n    path.write_text(\'{"uid":"a","status":"ok"}\\n{"uid":\')\n    rows = read_rows(path)\n    assert len(rows) == 1\n    rewrite_rows(path, rows)\n    assert json.loads(path.read_text())["uid"] == "a"\n    assert fingerprint({"a": 1, "b": 2}) == fingerprint({"b": 2, "a": 1})\n    assert fingerprint({"a": 1}) != fingerprint({"a": 2})\n\n\ndef test_tiles_respect_budget_and_keep_thumbnail():\n    image = Image.new("RGB", (1200, 1800))\n    tiles = intern_tiles(image, max_tiles=6)\n    assert 1 <= len(tiles) <= 7\n    assert all(t.size == (448, 448) for t in tiles)\n\n\ndef test_report_marks_incomplete(tmp_path):\n    from oprb_lab.report import summarize\n    (tmp_path / "manifest.json").write_text(json.dumps([{"uid": "a"}, {"uid": "b"}]))\n    row = dict(uid="a", status="ok", category="text", occlusion_type="Ink", density="2",\n               possibly_truncated=None, latency_sec=1, **score("cat", "cat"))\n    rewrite_rows(tmp_path / "rank0.jsonl", [row])\n    result = summarize(tmp_path)\n    assert result["missing"] == 1\n    assert not result["complete"]\n    assert result["truncation_unknown"] == 1\n    assert (tmp_path / "per_image_results.csv").exists()\n\n\ndef test_report_when_all_workers_fail_loading(tmp_path):\n    from oprb_lab.report import summarize\n    (tmp_path / "manifest.json").write_text(json.dumps([{"uid": "a"}]))\n    result = summarize(tmp_path)\n    assert result["missing"] == 1\n    assert result["success"] == 0\n    assert not result["complete"]\n    assert (tmp_path / "summary.json").exists()\n\n\ndef test_report_counts_failed_pages(tmp_path):\n    from oprb_lab.report import summarize\n    (tmp_path / "manifest.json").write_text(json.dumps([{"uid": "a"}]))\n    rewrite_rows(tmp_path / "rank0.jsonl", [dict(uid="a", status="error", error="OOM")])\n    result = summarize(tmp_path)\n    assert result["errors"] == 1\n    assert result["missing"] == 0\n    assert not result["complete"]\n', 'tests/test_notebook.py': '"""Notebook packaging checks; runnable without third-party dependencies."""\nimport ast\nimport json\nfrom pathlib import Path\nimport unittest\n\n\nclass NotebookTests(unittest.TestCase):\n    def test_five_primary_notebooks_use_shared_source(self):\n        import importlib.util\n        root = Path(__file__).resolve().parents[1]\n        spec = importlib.util.spec_from_file_location("generator", root / "generate_notebooks.py")\n        generator = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(generator)\n        self.assertEqual(set(generator.NAMES), {"paddle", "granite", "got", "internvl", "qwen"})\n        for key, filename in generator.NAMES.items():\n            nb = json.loads((root / filename).read_text(encoding="utf-8"))\n            for cell in nb["cells"]:\n                if cell["cell_type"] == "code":\n                    ast.parse("".join(cell["source"]))\n            setup = "".join(nb["cells"][1]["source"])\n            tree = ast.parse(setup)\n            assignments = {node.targets[0].id: node.value for node in tree.body\n                           if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name)}\n            self.assertEqual(ast.literal_eval(assignments["MODEL"]), key)\n            payload = ast.literal_eval(assignments["FILES"])\n            for path in (root / "src").rglob("*.py"):\n                self.assertEqual(payload[path.relative_to(root).as_posix()], path.read_text(encoding="utf-8"))\n        for filename in generator.LEGACY:\n            self.assertFalse((root / filename).exists())\n            self.assertTrue((root / "notebooks/legacy" / filename).exists())\n\n    def test_standalone_sources_and_valid_cells(self):\n        root = Path(__file__).resolve().parents[1]\n        notebook = json.loads((root / "notebooks/OPRB_Kaggle_2GPU.ipynb").read_text(encoding="utf-8"))\n        code = ["".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code"]\n        for source in code:\n            ast.parse(source)\n        tree = ast.parse(code[0])\n        payload = next(ast.literal_eval(n.value) for n in tree.body\n                       if isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "FILES" for t in n.targets))\n        for folder, pattern in (("src", "*.py"), ("requirements", "*.txt"), ("tests", "test_*.py")):\n            for path in (root / folder).rglob(pattern):\n                self.assertEqual(payload[path.relative_to(root).as_posix()], path.read_text(encoding="utf-8"))\n        self.assertIn(\'GPUS = "0,1"\', "".join(code))\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'requirements/internvl.txt': '# Legacy remote-code checkpoint: use a separate Kaggle session from Qwen3.\ntransformers==4.48.3\naccelerate>=1,<2\neinops\ntimm>=1,<2\nsentencepiece\n', 'requirements/paddle.txt': '# Install the CUDA PaddlePaddle wheel separately for your Kaggle CUDA environment.\npaddleocr==3.2.0\n', 'requirements/vlm.txt': '# Kaggle provides CUDA-enabled torch/torchvision. Keep its matching pair.\ntransformers==4.57.1\naccelerate>=1,<2\ndocling-core>=2.48,<3\neinops\ntimm>=1,<2\nsentencepiece\n'}
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content") if Path("/content").exists() else Path.cwd()
ROOT = BASE / "oprb-lab"
ROOT.mkdir(parents=True, exist_ok=True)
for name, content in FILES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
MODEL = 'granite'  # paddle, granite, got, internvl, qwen
LIMIT = 5
DATA = BASE / "oprb-data"
OUTPUT = BASE / "oprb-results"
GPUS = "0"  # Change to "0,1" on Kaggle T4 x2
ENV = dict(os.environ)
ENV.pop("LD_LIBRARY_PATH", None)
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, env=ENV, check=True)


In [ ]:
assert MODEL in ("paddle", "granite", "got", "internvl", "qwen")
run("-m", "pip", "install", "-e", ".[test]")
requirement = "paddle" if MODEL == "paddle" else "internvl" if MODEL == "internvl" else "vlm"
run("-m", "pip", "install", "-r", f"requirements/{requirement}.txt")
if MODEL != "paddle":
    run("-m", "pip", "install", "-U", "nvidia-nccl-cu12")
if MODEL == "paddle":
    # Change this index if the Kaggle CUDA environment requires another wheel.
    run("-m", "pip", "install", "paddlepaddle-gpu==3.2.0", "-i",
        "https://www.paddlepaddle.org.cn/packages/stable/cu126/")
run("-m", "pytest", "-q", "tests/test_core.py", "tests/test_comparison.py")


In [ ]:
run("-c", "import torch; print(torch.__version__); print(torch.cuda.get_device_name(0)); "
    f"assert torch.cuda.device_count() >= {len(GPUS.split(chr(44)))}, 'Enable the selected GPUs'")
if MODEL == "paddle":
    run("-c", "import paddle; assert paddle.is_compiled_with_cuda(); "
        f"assert paddle.device.cuda.device_count() >= {len(GPUS.split(chr(44)))}")
run("-m", "oprb_lab", "download", "--data", DATA)


In [ ]:
args = ["-m", "oprb_lab", "run", "--model", MODEL, "--data", DATA,
        "--output", OUTPUT, "--gpus", GPUS]
if LIMIT is not None:
    args += ["--limit", str(LIMIT)]
run(*args)  # Repeat this cell to resume the same configuration.


In [ ]:
import shutil
for path in sorted(OUTPUT.glob("*/summary.json")):
    print(path.parent.name)
    print(path.read_text())
archive = shutil.make_archive(str(BASE / "oprb-results"), "zip", OUTPUT)
from IPython.display import FileLink, display
display(FileLink(archive))
